
# Entanglement Links vs. Springs

In polymer network elasticity simulations using the Maximum Entropy Homogenization
Procedure (MEHP) (:cite:t:`gusev_numerical_2019`, :cite:t:`bernhard_phantom_2025`),
topological entanglements impose constraints on polymer chain fluctuations.

In :class:`~pylimer_tools_cpp.MEHPForceBalance2`, users can choose between two distinct
representations for sampled entanglements via the ``entanglements_as_springs`` parameter:

1. **Entanglement Links** (``entanglements_as_springs=False``, default):
   The two entangled strand beads are merged into a single shared 4-functional junction
   node ($\mathbf{r}_1 = \mathbf{r}_2$), equivalent to an instantaneous tetrafunctional cross-link.
   This imposes a rigid spatial constraint, effectively pinning the chains together at a localized junction.
   Importantly, in Force Balance 2, these links **cannot slip** along the chain contour; they remain
   fixed at their sampled contour positions.
2. **Entanglement Springs** (``entanglements_as_springs=True``):
   The two entangled strand beads remain distinct and are coupled by an additional
   Hookean spring. This introduces internal elastic compliance between the chains,
   allowing them to fluctuate at finite separation distances. Like entanglement links,
   entanglement springs **cannot slip** along the chain contour.

<div class="alert alert-info"><h4>Note</h4>

**Nomenclature Clarification**: In earlier literature and colloquial discussion, these models
have sometimes been loosely called "slip-links" or "slip-springs". However, in
:class:`~pylimer_tools_cpp.MEHPForceBalance2`, the links and springs **do not slip** along the
polymer chains. In :cite:t:`bernhard_phantom_2025`, they are referred to simply as
**entanglement links** (or entanglement beads) and **entanglement springs**.
The first-generation :class:`~pylimer_tools_cpp.MEHPForceBalance` did provide an option for
links to slide dynamically along the contour (``disable_slipping=False``), but slipping was
found to artificially soften the network modulus and add unnecessary numerical complexity.
For a detailed comparison of actual slipping vs. fixed links, see :doc:`plot_slipping_vs_fixed_links`.

</div>

This example illustrates:

- **Topological Differences**: Number of extra nodes, degrees of freedom, and springs introduced.
- **Elastic Compliance & Scaling**: How the shear modulus $G$ scales with entanglement density under both models.
- **Statistical Significance**: Multi-seed ensemble sampling with 95% confidence intervals.
- **Physical Guidance**: When to choose entanglement links versus entanglement springs.


## Background & Mathematical Formulation
When an entanglement is identified between two strands $s_1$ and $s_2$ at
contour positions $\alpha_1$ and $\alpha_2$:

- In the **Entanglement Link** representation:
  A single node is added at position $\mathbf{R}_{\text{link}} = \frac{1}{2}(\mathbf{r}_1 + \mathbf{r}_2)$.
  Both strands are partitioned at this node into sub-springs:

  .. math::

     N_{\text{extra nodes}} = N_{\text{ent}}, \quad N_{\text{extra springs}} = 0

  Because the constraint is rigid, any force transmitted across the entanglement directly
  stresses the network backbone without internal deflection.

- In the **Entanglement Spring** representation:
  Two separate nodes are placed at $\mathbf{r}_1$ and $\mathbf{r}_2$, and an
  explicit Hookean spring connects them:

  .. math::

     N_{\text{extra nodes}} = 2 N_{\text{ent}}, \quad N_{\text{extra springs}} = N_{\text{ent}}

  The entanglement spring acts in series with the strand segments, adding compliance and
  reducing the effective stiffness of the entanglement constraint.



In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
import sys
from typing import Any, Dict, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np

# Ensure examples/ is on sys.path when script is executed directly
if "__file__" in globals():
    _examples_dir = str(Path(__file__).resolve().parents[1])
    if _examples_dir not in sys.path:
        sys.path.insert(0, _examples_dir)

from _example_cache import (  # noqa: E402
    compute_ensemble_statistics,
    configure_process_threading,
    get_example_cache_path,
    load_or_compute_cache,
    resolve_max_workers,
    worker_process_initializer,
)
from pylimer_tools.io.bead_spring_parameter_provider import (  # noqa: E402
    ParameterType,
    get_parameters_for_polymer,
)
from pylimer_tools_cpp import (  # noqa: E402
    EntanglementFilteringMode,
    MCUniverseGenerator,
)

## Multi-Seed Simulation Procedure
To ensure statistically meaningful conclusions, we generate independent Monte Carlo
network realizations across multiple random seeds and sample entanglements stochastically.

To keep CI execution and documentation builds fast (< 1 s) while allowing deep statistical
sampling (multiple seeds and complex systems), the results are cached into a committed
``.npz`` archive and loaded automatically. You can rerun the live multi-seed computation
at any time by passing ``--recompute`` or setting
``PYLIMER_RECOMPUTE_CACHE=1``.



In [ ]:
def _simulate_link_vs_spring_single_repeat(
    r: int,
    seed: int,
    box_l: float,
    b: float,
    n_strands: int,
    strand_len: int,
    conversion: float,
    density_factors: Sequence[float],
    nominal_n_ent: int,
    cutoff: float,
    r02: float,
    vol: float,
    gamma_conv: float,
) -> Tuple[int, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Execute a single Monte Carlo network realization across entanglement densities."""
    gen = MCUniverseGenerator(box_l, box_l, box_l)
    gen.set_seed(seed)
    gen.set_bead_distance(b)
    gen.set_mean_squared_bead_distance(b**2)
    gen.add_strands(n_strands, [strand_len] * n_strands)
    gen.add_crosslinkers(int(0.5 * n_strands), 4)
    gen.link_strands_to_conversion(conversion)

    n_factors = len(density_factors)
    g_links = np.zeros(n_factors)
    g_springs = np.zeros(n_factors)
    extra_links = np.zeros(n_factors)
    extra_springs = np.zeros(n_factors)

    for i, factor in enumerate(density_factors):
        n_target = int(factor * nominal_n_ent)

        if n_target == 0:
            # Bare unentangled crosslinker network
            fb_bare = gen.get_force_balance2()
            fb_bare.run_force_relaxation()
            g_pa = (gamma_conv * fb_bare.get_gamma_factor(r02) / vol) * 1e6
            g_links[i] = g_pa
            g_springs[i] = g_pa
            extra_links[i] = 0
            extra_springs[i] = 0
        else:
            # Sample entanglements using seeded Brownian bridge
            es = gen.sample_entanglements(
                n_target,
                cutoff,
                0.0,
                0,
                3.0,
                f"ent_{seed}_{factor}",
                filtering_mode=EntanglementFilteringMode.OMIT_INACTIVE,
            )

            # 1. Entanglement Links (merged junction nodes)
            fb_l = gen.get_force_balance2(
                es, crosslinker_type=2, entanglements_as_springs=False
            )
            extra_links[i] = fb_l.get_nr_of_extra_atoms()
            fb_l.run_force_relaxation()
            g_links[i] = (gamma_conv * fb_l.get_gamma_factor(r02) / vol) * 1e6

            # 2. Entanglement Springs (explicit Hookean coupling springs)
            fb_s = gen.get_force_balance2(
                es, crosslinker_type=2, entanglements_as_springs=True
            )
            extra_springs[i] = fb_s.get_nr_of_extra_atoms()
            fb_s.run_force_relaxation()
            g_springs[i] = (
                gamma_conv * fb_s.get_gamma_factor(r02) / vol) * 1e6

    return r, g_links, g_springs, extra_links, extra_springs


def run_entanglement_link_vs_spring_simulation(
    density_factors: Sequence[float] = (0.0, 0.5, 1.0, 1.5, 2.0),
    n_strands: int = 10000,
    strand_len: int = 50,
    conversion: float = 0.95,
    n_repeats: int = 10,
    base_seed: int = 123,
    max_workers: Optional[int] = None,
) -> Dict[str, Any]:
    """Evaluate shear modulus and extra degrees of freedom across entanglement densities.

    Averages over independent network realizations and stochastic entanglement samplings
    to provide statistically robust comparisons with Student's t 95% confidence intervals.
    """
    params = get_parameters_for_polymer(
        "PDMS", parameter_type=ParameterType.GAUSSIAN)
    b = params.get("<b>").to(params.get("distance_units")).magnitude
    bead_density = params.get_bead_density()
    cutoff = params.get_sampling_cutoff()
    nominal_ent_density = params.get_entanglement_density()
    r02 = params.get("R02").to(params.get("distance_units") ** 2).magnitude
    kbt = params.get("T") * params.get("kb")
    gamma_conv = (
        (kbt / ((params.get("distance_units")) ** 3)).to("MPa")).magnitude

    n_atoms = n_strands * strand_len + int(0.5 * n_strands)
    vol = n_atoms / bead_density
    box_l = vol ** (1.0 / 3.0)
    nominal_n_ent = int(nominal_ent_density * vol)

    all_g_links = np.zeros((len(density_factors), n_repeats))
    all_g_springs = np.zeros((len(density_factors), n_repeats))
    all_extra_links = np.zeros((len(density_factors), n_repeats))
    all_extra_springs = np.zeros((len(density_factors), n_repeats))

    tasks = [
        (
            r,
            base_seed + 101 * r,
            box_l,
            b,
            n_strands,
            strand_len,
            conversion,
            density_factors,
            nominal_n_ent,
            cutoff,
            r02,
            vol,
            gamma_conv,
        )
        for r in range(n_repeats)
    ]

    workers = min(resolve_max_workers(max_workers), n_repeats)
    configure_process_threading(workers)

    if workers > 1 and n_repeats > 1:
        print(
            f"Executing {n_repeats} repeats in parallel across {workers} worker processes..."
        )
        completed = 0
        with ProcessPoolExecutor(
            max_workers=workers, initializer=worker_process_initializer
        ) as executor:
            futures = [
                executor.submit(_simulate_link_vs_spring_single_repeat, *task)
                for task in tasks
            ]
            for fut in as_completed(futures):
                r, g_l, g_s, ex_l, ex_s = fut.result()
                all_g_links[:, r] = g_l
                all_g_springs[:, r] = g_s
                all_extra_links[:, r] = ex_l
                all_extra_springs[:, r] = ex_s
                completed += 1
                print(
                    f"  [Link vs Spring] Completed repeat {completed}/{
                        n_repeats
                    } (repeat index {r + 1})..."
                )
    else:
        for task in tasks:
            r, g_l, g_s, ex_l, ex_s = _simulate_link_vs_spring_single_repeat(
                *task)
            all_g_links[:, r] = g_l
            all_g_springs[:, r] = g_s
            all_extra_links[:, r] = ex_l
            all_extra_springs[:, r] = ex_s
            print(
                f"  [Link vs Spring] Completed repeat {
                    r + 1}/{n_repeats}...")

    # Compute statistical moments and 95% confidence intervals across seeds
    stat_links = compute_ensemble_statistics(all_g_links, axis=1)
    stat_springs = compute_ensemble_statistics(all_g_springs, axis=1)
    stat_extra_l = compute_ensemble_statistics(all_extra_links, axis=1)
    stat_extra_s = compute_ensemble_statistics(all_extra_springs, axis=1)

    return {
        "density_factors": np.array(density_factors),
        "n_repeats": np.array(n_repeats),
        "nominal_n_ent": np.array(nominal_n_ent),
        "vol": np.array(vol),
        "g_links_mean": stat_links["mean"],
        "g_links_std": stat_links["std"],
        "g_links_sem": stat_links["sem"],
        "g_links_ci": stat_links["ci"],
        "g_links_all": all_g_links,
        "g_springs_mean": stat_springs["mean"],
        "g_springs_std": stat_springs["std"],
        "g_springs_sem": stat_springs["sem"],
        "g_springs_ci": stat_springs["ci"],
        "g_springs_all": all_g_springs,
        "extra_atoms_links_mean": stat_extra_l["mean"],
        "extra_atoms_links_std": stat_extra_l["std"],
        "extra_atoms_springs_mean": stat_extra_s["mean"],
        "extra_atoms_springs_std": stat_extra_s["std"],
    }

## Loading Cached Data or Recomputing
Resolves the committed cache file or recomputes when requested:



In [ ]:
if __name__ == "__main__" or "sphinx_gallery" in sys.modules:
    cache_path = get_example_cache_path(
        "entanglement_link_vs_spring_cache.npz", "force_balance"
    )
    data = load_or_compute_cache(cache_path,
                                 run_entanglement_link_vs_spring_simulation)

    density_factors = data["density_factors"]
    n_repeats = int(data.get("n_repeats", 5))

    g_links_mean = data["g_links_mean"]
    g_links_ci = data["g_links_ci"]

    g_springs_mean = data["g_springs_mean"]
    g_springs_ci = data["g_springs_ci"]

    extra_links_mean = data["extra_atoms_links_mean"]
    extra_springs_mean = data["extra_atoms_springs_mean"]

## Visualizing Mechanical & Structural Differences
The plots below compare:

1. **Equilibrium Shear Modulus $G$ vs. Entanglement Density**:
   Entanglement links exhibit a strong, near-linear increase in modulus with entanglement density
   due to rigid node coalescing. In contrast, entanglement springs display a softer, sublinear
   response that saturates because the intermediate spring provides series elastic compliance.
   Shaded error bands and error bars represent the **95% Student's t confidence interval**
   computed across independent network and sampling realizations.
2. **System Degrees of Freedom (Extra Nodes)**:
   Entanglement springs introduce exactly twice as many extra node coordinates into the linear
   system of equations as entanglement links.



In [ ]:
if __name__ == "__main__" or "sphinx_gallery" in sys.modules:
    # Figure 1: Modulus comparison with shaded 95% confidence intervals and
    # error bars
    fig, ax_mod = plt.subplots(figsize=(7, 4.5))
    ax_mod.plot(
        density_factors,
        g_links_mean,
        "s-",
        color="#d95f02",
        linewidth=2,
        markersize=6,
        label=f"Entanglement Links (Merged Nodes, N={n_repeats})",
    )
    ax_mod.fill_between(
        density_factors,
        g_links_mean - g_links_ci,
        g_links_mean + g_links_ci,
        color="#d95f02",
        alpha=0.2,
        label="95% Confidence Interval",
    )
    ax_mod.errorbar(
        density_factors,
        g_links_mean,
        yerr=g_links_ci,
        fmt="none",
        ecolor="#d95f02",
        capsize=4,
        elinewidth=1.2,
    )

    ax_mod.plot(
        density_factors,
        g_springs_mean,
        "o-",
        color="#1b9e77",
        linewidth=2,
        markersize=6,
        label=f"Entanglement Springs (Hookean Springs, N={n_repeats})",
    )
    ax_mod.fill_between(
        density_factors,
        g_springs_mean - g_springs_ci,
        g_springs_mean + g_springs_ci,
        color="#1b9e77",
        alpha=0.2,
    )
    ax_mod.errorbar(
        density_factors,
        g_springs_mean,
        yerr=g_springs_ci,
        fmt="none",
        ecolor="#1b9e77",
        capsize=4,
        elinewidth=1.2,
    )

    ax_mod.set_xlabel(
        r"Entanglement Density Factor ($N_{\mathrm{ent}} / N_{\mathrm{nominal}}$)"
    )
    ax_mod.set_ylabel(r"Shear Modulus $G$ [Pa]")
    ax_mod.set_title("Equilibrium Shear Modulus vs. Entanglement Density")
    ax_mod.grid(True, linestyle="--", alpha=0.5)
    ax_mod.legend(frameon=True)
    plt.tight_layout()
    plt.show()

    # Figure 2: System size comparison
    fig, ax_dof = plt.subplots(figsize=(7, 4.5))
    ax_dof.plot(
        density_factors,
        extra_links_mean,
        "s-",
        color="#d95f02",
        linewidth=2,
        markersize=6,
        label=r"Entanglement Links: $N_{\mathrm{extra}} = N_{\mathrm{ent}}$",
    )
    ax_dof.plot(
        density_factors,
        extra_springs_mean,
        "o-",
        color="#1b9e77",
        linewidth=2,
        markersize=6,
        label=r"Entanglement Springs: $N_{\mathrm{extra}} = 2 N_{\mathrm{ent}}$",
    )

    ax_dof.set_xlabel(
        r"Entanglement Density Factor ($N_{\mathrm{ent}} / N_{\mathrm{nominal}}$)"
    )
    ax_dof.set_ylabel("Extra Nodes Added to SLE")
    ax_dof.set_title("System Size / Extra Degrees of Freedom")
    ax_dof.grid(True, linestyle="--", alpha=0.5)
    ax_dof.legend(frameon=True)

    plt.tight_layout()
    plt.show()

## Practical Recommendations

.. list-table::
   :widths: 25 35 40
   :header-rows: 1

   * - Feature
     - Entanglement Links (``entanglements_as_springs=False``)
     - Entanglement Springs (``entanglements_as_springs=True``)
   * - **Constraint Nature**
     - Rigid point constraint ($\mathbf{r}_1 = \mathbf{r}_2$).
     - Compliant harmonic coupling ($k_{\text{spring}} > 0$).
   * - **Modulus Response**
     - Higher; scales more steeply with entanglement density.
     - Softer; exhibits saturation due to series compliance.
   * - **System Size (SLE)**
     - Smaller ($+ N_{\text{ent}}$ nodes, $0$ extra bonds).
     - Larger ($+ 2 N_{\text{ent}}$ nodes, $+ N_{\text{ent}}$ extra springs).
   * - **Recommended Use**
     - Standard affine/phantom homogenization where entanglements act as effective crosslinks (:cite:t:`bernhard_phantom_2025`).
     - Studies exploring inter-chain interaction compliance or finite-range topological coupling.

